In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn import datasets
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

np.random.seed(42)
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 1. 读取数据集

In [ ]:
iris = datasets.load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print("样本数:", X.shape[0])
print("特征数:", X.shape[1])
print("类别:", list(target_names))
print("前 5 条样本:\n", X[:5])

## 2. 数据可视化

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(7, 5), dpi=150)
for idx, name in enumerate(target_names):
    mask = y == idx
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=name, alpha=0.85)

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Iris 数据集 PCA 可视化")
plt.legend()
plt.tight_layout()
plt.show()

## 3. 训练 SVM 分类器

SVM 对特征尺度比较敏感，因此这里先使用 `StandardScaler` 做标准化，再训练 `SVC`。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(kernel="rbf", C=1.0, gamma="scale")),
])

svm_clf.fit(X_train, y_train)
y_pred = svm_clf.predict(X_test)

## 4. 模型评估

In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"测试集准确率: {acc:.4f}")
print("\n分类报告:\n")
print(classification_report(y_test, y_pred, target_names=target_names))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)

fig, ax = plt.subplots(figsize=(5, 4), dpi=150)
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("SVM 混淆矩阵")
plt.tight_layout()
plt.show()

## 5. 简单调参

这里使用网格搜索在训练集上尝试不同的 `C` 和 `gamma`，观察是否能进一步提升效果。

In [ ]:
param_grid = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__gamma": ["scale", 0.1, 0.01, 0.001],
    "svc__kernel": ["rbf"],
}

grid = GridSearchCV(svm_clf, param_grid=param_grid, cv=5)
grid.fit(X_train, y_train)

print("最优参数:", grid.best_params_)
print(f"交叉验证最高分数: {grid.best_score_:.4f}")

best_model = grid.best_estimator_
best_pred = best_model.predict(X_test)
print(f"调参后测试集准确率: {accuracy_score(y_test, best_pred):.4f}")